# Exploratory Data Analysis — Magnetic Tile Surface Defects

This notebook is exploratory only. All logic used for the actual pipeline lives in `src/`; this notebook simply calls into `src/data/analysis.py` and `src/data/dataset.py` and visualizes the results inline.

In [ ]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config_loader import load_config
from src.data.dataset import scan_dataset, stratified_split
from src.data.analysis import (compute_class_distribution, plot_class_distribution,
                                plot_sample_grid, compute_image_size_stats)

config = load_config(os.path.abspath(os.path.join(os.getcwd(), '..', 'config', 'config.yaml')))
config['data']['raw_dir']

In [ ]:
samples = scan_dataset(
    raw_dir=config['data']['raw_dir'],
    classes=config['classes'],
    class_folder_prefix=config['class_folder_prefix'],
    image_extension=config['data']['image_extension'],
)
print(f"Total images found: {len(samples)}")

## Class distribution

The Magnetic Tile Surface Defects dataset is naturally imbalanced across its six classes (five defect types plus a defect-free class). This is the imbalance that our weighted-loss strategy in `src/training/utils.py` is designed to address.

In [ ]:
distribution = compute_class_distribution(samples, config['classes'])
for class_name, count in distribution.items():
    print(f"{class_name:12s}: {count}")

plot_class_distribution(samples, config['classes'],
                         os.path.join(config['evaluation']['figures_dir'], 'class_distribution.png'))

## Sample images per class

Visually inspecting a few samples per class helps confirm that classes are visually distinguishable and surfaces any obviously mislabeled or ambiguous images before training.

In [ ]:
plot_sample_grid(samples, config['classes'],
                  os.path.join(config['evaluation']['figures_dir'], 'sample_grid.png'),
                  samples_per_class=3)

## Image size statistics

This justifies the preprocessing decision in `src/data/transforms.py`: since raw images vary in resolution/aspect ratio, we resize (not center-crop) to avoid cutting off defects near tile edges.

In [ ]:
stats = compute_image_size_stats(samples)
stats

## Stratified split sanity check

Confirms that the train/val/test split preserves class proportions from the full dataset.

In [ ]:
train_set, val_set, test_set = stratified_split(
    samples,
    train_split=config['data']['train_split'],
    val_split=config['data']['val_split'],
    test_split=config['data']['test_split'],
    seed=config['seed'],
)

for split_name, split_data in [('train', train_set), ('val', val_set), ('test', test_set)]:
    dist = compute_class_distribution(split_data, config['classes'])
    print(f"\n{split_name} (n={len(split_data)}):")
    for class_name, count in dist.items():
        pct = 100 * count / len(split_data)
        print(f"  {class_name:12s}: {count:4d} ({pct:.1f}%)")